# Bài học 05 - RAG có tác nhân


## Thiết lập

Notebook này trình bày mẫu Agentic RAG (Retrieval-Augmented Generation) sử dụng Microsoft Agent Framework.

**Yêu cầu trước:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — điểm cuối dịch vụ Azure AI Search của bạn
- `AZURE_SEARCH_API_KEY` — khóa API Azure AI Search của bạn
- Triển khai Azure OpenAI được cấu hình qua biến môi trường
- Đã xác thực Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG là gì?

RAG truyền thống tuân theo một quy trình cố định: truy xuất tài liệu, sau đó tạo câu trả lời. **Agentic RAG** đi xa hơn bằng cách trao cho tác nhân quyền tự chủ để quyết định **khi nào** và **cách thức** truy xuất thông tin.

Với Agentic RAG, tác nhân có thể:
- **Quyết định** xem có cần truy xuất trước khi trả lời câu hỏi hay không
- **Chọn** nguồn dữ liệu hoặc công cụ để truy vấn
- **Đánh giá** kết quả đã truy xuất và thực hiện truy xuất tiếp theo nếu lần đầu không đủ
- **Kết hợp** thông tin từ nhiều bước truy xuất thành một câu trả lời mạch lạc

Điều này làm cho tác nhân linh hoạt và chính xác hơn so với quy trình truy xuất-rồi-tạo câu trả lời tĩnh.


## Tạo công cụ tìm kiếm

Trong Agentic RAG, các nguồn dữ liệu bên ngoài được đóng gói như các **công cụ** mà tác nhân có thể gọi khi cần. Điều này cho phép tác nhân xem việc truy xuất như một hành động khác mà nó có thể thực hiện, thay vì một bước bắt buộc.

Dưới đây chúng tôi định nghĩa một cơ sở kiến thức về du lịch và cung cấp nó như một công cụ mà tác nhân có thể gọi để tra cứu thông tin điểm đến.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Xây dựng tác nhân RAG

Bây giờ chúng ta tạo một tác nhân được chỉ định **luôn truy xuất thông tin trước khi trả lời**. Tác nhân sử dụng công cụ `search_travel_knowledge` để căn cứ câu trả lời của mình vào cơ sở tri thức thay vì dựa vào dữ liệu huấn luyện của chính nó.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Truy xuất Lặp lại — Mẫu Maker-Checker

Một lợi thế chính của Agentic RAG là **truy xuất lặp lại**. Đại lý có thể thực hiện nhiều vòng tìm kiếm để xác minh, chỉnh sửa hoặc mở rộng các phát hiện ban đầu — tương tự như quy trình làm việc "maker-checker":

1. **Bước Maker**: Đại lý truy xuất thông tin ban đầu và soạn thảo câu trả lời.
2. **Bước Checker**: Đại lý thực hiện các truy xuất bổ sung để xác minh chi tiết hoặc lấp đầy các khoảng trống.

Dưới đây, đại lý được hỏi một câu hỏi yêu cầu so sánh nhiều điểm đến, khiến nó phải tìm kiếm nhiều lần.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Tóm tắt

Trong bài học này, bạn đã học cách xây dựng một hệ thống **Agentic RAG** sử dụng Microsoft Agent Framework:

- **Agentic RAG** cho phép các agent quyết định một cách tự động khi nào cần truy xuất thông tin, khiến việc truy xuất trở nên linh hoạt thay vì cố định.
- **Công cụ như nguồn dữ liệu**: Các cơ sở tri thức bên ngoài (như Azure AI Search) được đóng gói như các công cụ mà agent có thể gọi.
- **Truy xuất lặp đi lặp lại**: Mô hình maker-checker cho phép agent thực hiện nhiều vòng truy xuất — tìm kiếm, xác minh và tinh chỉnh — trước khi tạo ra câu trả lời cuối cùng.

Trong môi trường sản xuất, bạn sẽ thay thế `TRAVEL_KNOWLEDGE_BASE` trong bộ nhớ bằng một chỉ mục Azure AI Search thật để xử lý việc truy xuất tài liệu du lịch quy mô lớn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trách**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc thiếu chính xác. Tài liệu gốc bằng ngôn ngữ nguyên thủy nên được coi là nguồn thông tin chính thức. Đối với thông tin quan trọng, chúng tôi khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu nhầm hay giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
